# ✦ LILY WAN 2.2 — Adaptive Kaggle Studio — v5

Choose **GPU T4 x2** if Kaggle honors it. If Kaggle overrides you with a **P100**, this automatically switches to a conservative single-GPU profile. This version does **not** use Python venv; ComfyUI extras are installed into an isolated `/kaggle/working/lily_pkgs` folder with `pip --target`, so Kaggle's notebook kernel is left alone.


In [ ]:
import json, urllib.request, sys, subprocess, os
from pathlib import Path

print('✦ Lily Wan 2.2 Studio — adaptive startup v5')

# ------------------------------------------------------------
# 0) Detect what Kaggle actually gave us.
# ------------------------------------------------------------
try:
    raw = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        text=True, stderr=subprocess.STDOUT
    )
    gpu_names = [x.strip() for x in raw.splitlines() if x.strip()]
except Exception as e:
    raise RuntimeError(f'Could not detect Kaggle GPUs: {e}')

print('Detected GPUs:', gpu_names)
is_dual_t4 = len(gpu_names) >= 2 and all('T4' in n.upper() for n in gpu_names[:2])
is_p100 = len(gpu_names) >= 1 and 'P100' in gpu_names[0].upper()
if is_dual_t4:
    PROFILE = 'DUAL_T4'
    print('✓ Profile: DUAL T4 — preferred/full profile')
elif is_p100:
    PROFILE = 'P100'
    print('✓ Profile: SINGLE P100 — conservative 16 GB fallback')
else:
    PROFILE = 'GENERIC_16GB'
    print('⚠ Unknown GPU; using conservative 16 GB fallback')

# ------------------------------------------------------------
# 1) Isolated package TARGET — no venv, no kernel mutation.
# ------------------------------------------------------------
PKG_TARGET = Path('/kaggle/working/lily_pkgs')
PKG_TARGET.mkdir(parents=True, exist_ok=True)
print('✓ Package sandbox:', PKG_TARGET)

# ------------------------------------------------------------
# 2) Load the known full studio snapshot and patch it.
# ------------------------------------------------------------
SOURCE_URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/789b857e85738efdaec591f04de11bf76befe20d/LILY_WAN22_DUAL_T4_STUDIO.ipynb'
with urllib.request.urlopen(SOURCE_URL, timeout=60) as r:
    original = json.loads(r.read().decode('utf-8'))
cells = [c for c in original['cells'] if c.get('cell_type') == 'code']
if not cells:
    raise RuntimeError('Could not locate the studio code cell.')
code = ''.join(cells[0]['source'])

# Never blanket-upgrade Kaggle's requests/pandas/jupyter/etc.
bad_core = '''# Core helpers/UI.
pip_install("gradio>=5.20,<6", "huggingface_hub>=0.29", "requests>=2.32", "gdown>=5.2", "scikit-video")'''
safe_core = '''# Core helpers/UI — keep Kaggle kernel untouched.
print("✓ Using Kaggle's existing notebook packages — no core upgrades")'''
if bad_core not in code:
    raise RuntimeError('Could not locate unsafe core installer.')
code = code.replace(bad_core, safe_core, 1)

# Defuse any helper installs: extras go into lily_pkgs, never site-packages.
old_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])'''
new_helper = '''def pip_install(*pkgs):
    run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "--target", str(PKG_TARGET), *pkgs], check=False)'''
if old_helper in code:
    code = code.replace(old_helper, new_helper, 1)

# Replace ComfyUI's direct requirements install with a filtered --target install.
old_req = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=COMFY)'
new_req = '''print("Installing ComfyUI extras into isolated lily_pkgs folder...")
_req_src = COMFY / "requirements.txt"
_req_safe = COMFY / "requirements_lily_safe.txt"
_skip = ("torch", "torchvision", "torchaudio")
_lines = []
for _line in _req_src.read_text().splitlines():
    _s = _line.strip()
    if not _s or _s.startswith("#"):
        _lines.append(_line)
        continue
    _name = _s.split(";", 1)[0].strip().lower()
    if any(_name == x or _name.startswith(x + "==") or _name.startswith(x + ">=") or _name.startswith(x + "<=") or _name.startswith(x + "~=") for x in _skip):
        print("  reusing Kaggle system package instead of reinstalling:", _s)
        continue
    _lines.append(_line)
_req_safe.write_text("\n".join(_lines) + "\n")
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "--target", str(PKG_TARGET), "-r", str(_req_safe)], cwd=COMFY, check=True)
print("✓ ComfyUI extras installed without touching the notebook kernel")'''
if old_req not in code:
    raise RuntimeError('Could not locate ComfyUI requirements installer.')
code = code.replace(old_req, new_req, 1)

# MagCache extras also go into the target folder.
old_mag = 'run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)'
new_mag = 'run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", "--prefer-binary", "--target", str(PKG_TARGET), "-r", str(req)], check=False)'
if old_mag in code:
    code = code.replace(old_mag, new_mag, 1)

# Child ComfyUI process sees lily_pkgs first, while notebook kernel does not.
old_cuda_env = '    env["CUDA_VISIBLE_DEVICES"] = "0"'
new_cuda_env = '''    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONPATH"] = str(PKG_TARGET) + os.pathsep + env.get("PYTHONPATH", "")'''
if old_cuda_env not in code:
    raise RuntimeError('Could not locate ComfyUI environment setup.')
code = code.replace(old_cuda_env, new_cuda_env, 1)

# Skip Practical-RIFE during startup.
rife_start = '# -------------------------\n# 4) Optional RIFE on GPU 1'
rife_end = '# -------------------------\n# 5) Start ComfyUI on GPU 0'
a, b = code.find(rife_start), code.find(rife_end)
if a == -1 or b == -1 or b <= a:
    raise RuntimeError('Could not locate RIFE startup block.')
no_rife = '''# -------------------------
# 4) RIFE deferred for reliable startup
# -------------------------
RIFE_READY = False
print("✓ RIFE deferred — ffmpeg interpolation enabled")

'''
code = code[:a] + no_rife + code[b:]

# P100 / generic 16 GB profile: lower native workload automatically.
if PROFILE != 'DUAL_T4':
    old_presets = '''PRESETS = {
    # Frame counts are 4k+1, as Wan expects.
    # Lower fps makes each native sequence ~5 seconds before interpolation.
    "⚡ Turbo":  {"frames": 61,  "steps": 12, "source_fps": 12.0, "cfg": 5.0},
    "✨ Normal": {"frames": 81,  "steps": 16, "source_fps": 16.0, "cfg": 5.0},
    "👑 Max":    {"frames": 121, "steps": 20, "source_fps": 24.0, "cfg": 5.0},
}
SIZES = {
    "Landscape 16:9": (1280, 704),
    "Portrait 9:16": (704, 1280),
}'''
    safe_presets = '''PRESETS = {
    "⚡ Turbo":  {"frames": 49, "steps": 10, "source_fps": 10.0, "cfg": 5.0},
    "✨ Normal": {"frames": 61, "steps": 12, "source_fps": 12.0, "cfg": 5.0},
    "👑 Max":    {"frames": 81, "steps": 16, "source_fps": 16.0, "cfg": 5.0},
}
SIZES = {
    "Landscape 16:9": (832, 480),
    "Portrait 9:16": (480, 832),
}'''
    if old_presets not in code:
        raise RuntimeError('Could not locate preset block for 16 GB fallback.')
    code = code.replace(old_presets, safe_presets, 1)

print(f'✓ Hardware profile selected: {PROFILE}')
print('✓ No venv required')
print('✓ Dependency sandbox patch applied')
print('✓ Starting Wan 2.2 setup...\n')
exec(compile(code, 'LILY_WAN22_ADAPTIVE_V5', 'exec'), globals(), globals())
